# Waynex Neural Engine - Advanced Deep Reinforcement Learning Training
This notebook trains the GNN-RL routing policy on Google Colab Pro using PPO (Return-to-Go).
**Instructions:**
1. Upload `drl_policy.py` and `gnn_encoder.py` to your Colab workspace sidebar.
2. Run the cells below to start training on the L4/A100 GPU.
3. Download the resulting `waynex_policy_v2.pt` file when finished.

In [ ]:
!pip install torch numpy

In [ ]:
# train_colab_v2.py
# Waynex Neural Engine - Advanced Deep Reinforcement Learning Training
# Upload this script to Colab along with drl_policy.py and gnn_encoder.py
# Run with: !python train_colab_v2.py

import torch
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time

from drl_policy import WaynexActorCriticPolicy
from gnn_encoder import prepare_graph_node_features, create_fully_connected_edge_index

def train_advanced():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 Starting Waynex Advanced DRL Training on {device}...")

    # Initialize the Neural Network and Optimizer
    model = WaynexActorCriticPolicy(in_features=6, hidden_dim=64, embed_dim=64).to(device)
    optimizer = optim.Adam(model.parameters(), lr=5e-4)

    num_episodes = 50000
    print(f"📦 Commencing {num_episodes} synthetic rollout episodes...")

    model.train()
    for ep in range(1, num_episodes + 1):
        num_nodes = 16
        coords = [{"lat": 12.9 + np.random.rand()*0.1, "lng": 77.5 + np.random.rand()*0.1} for _ in range(num_nodes)]
        deliveries = [{"demand": np.random.randint(50, 150)} for _ in range(num_nodes - 1)]
        vehicles = [{"capacity": 500, "type": "diesel"} for _ in range(3)]
        
        node_features = prepare_graph_node_features(coords, deliveries).to(device)
        edge_index = create_fully_connected_edge_index(num_nodes).to(device)
        
        unvisited = set(range(1, num_nodes))
        vehicle_load = [0] * len(vehicles)
        
        log_probs = []
        values = []
        total_dist = 0.0
        
        for v_idx, v in enumerate(vehicles):
            curr_node = 0
            v_cap = v["capacity"]
            
            while unvisited:
                veh_state = torch.tensor([[vehicle_load[v_idx] / v_cap, 1.0, 0.0, 0.0]], dtype=torch.float32).to(device)
                action_mask = torch.zeros(num_nodes, dtype=torch.float32).to(device)
                has_valid = False
                
                for candidate in unvisited:
                    if vehicle_load[v_idx] + deliveries[candidate - 1]["demand"] <= v_cap:
                        action_mask[candidate] = 1.0
                        has_valid = True
                
                if not has_valid:
                    break
                    
                logits, masked_logits, probs, value = model(node_features, edge_index, curr_node, veh_state, action_mask)
                
                m = torch.distributions.Categorical(probs)
                action = m.sample()
                
                if action.item() not in unvisited or action_mask[action.item()] < 0.5:
                    valid_indices = torch.where(action_mask > 0.5)[0]
                    if len(valid_indices) == 0: break
                    action = valid_indices[0]
                    
                log_probs.append(m.log_prob(action))
                values.append(value)
                
                next_node = action.item()
                unvisited.remove(next_node)
                vehicle_load[v_idx] += deliveries[next_node - 1]["demand"]
                
                dist = np.sqrt((coords[curr_node]["lat"] - coords[next_node]["lat"])**2 + (coords[curr_node]["lng"] - coords[next_node]["lng"])**2)
                total_dist += dist
                curr_node = next_node
                
        penalty = len(unvisited) * 10.0
        reward = -(total_dist + penalty)
        
        # Calculate discounted return-to-go
        returns = []
        R = reward
        for step in reversed(range(len(log_probs))):
            returns.insert(0, R)
            R = R * 0.99  # gamma
            
        optimizer.zero_grad()
        if len(log_probs) > 0:
            log_probs = torch.stack(log_probs)
            values = torch.cat(values)
            returns = torch.tensor(returns, dtype=torch.float32, device=device)
            
            # Normalize returns for stability
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
            advantage = returns - values.detach()
            
            actor_loss = -(log_probs * advantage).mean()
            critic_loss = F.mse_loss(values, returns)
            loss = actor_loss + 0.5 * critic_loss
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()
            
        if ep % 1000 == 0:
            print(f"Episode {ep}/{num_episodes} - Reward: {reward:.2f} - Unvisited: {len(unvisited)}")

    save_path = "waynex_policy_v2.pt"
    torch.save(model.state_dict(), save_path)
    print(f"✅ Training Complete! Model checkpoint saved to {save_path}")

if __name__ == "__main__":
    train_advanced()


In [ ]:
train_advanced()